# 音频模型高级教程

本教程涵盖高级主题：CTC/RNN-T、Conformer、FastSpeech、数据增强、评估指标等。

## 目录
1. [CTC 损失与解码](#1-ctc-损失与解码)
2. [Conformer 架构](#2-conformer-架构)
3. [非自回归 TTS](#3-非自回归-tts)
4. [音频数据增强](#4-音频数据增强)
5. [评估指标](#5-评估指标)
6. [流式推理](#6-流式推理)

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)

## 1. CTC 损失与解码

CTC (Connectionist Temporal Classification) 允许输入输出序列长度不同。

In [ ]:
class CTCModel(nn.Module):
    """简单的 CTC 语音识别模型"""
    def __init__(self, input_dim=80, hidden_dim=256, vocab_size=50):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, num_layers=3,
                               batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)
    
    def forward(self, x):
        out, _ = self.encoder(x)
        logits = self.fc(out)
        return F.log_softmax(logits, dim=-1)

# 测试
model = CTCModel().to(device)
x = torch.randn(4, 100, 80).to(device)  # [B, T, F]
log_probs = model(x)
print(f'CTC 输出: {log_probs.shape}')

# CTC 损失
ctc_loss = nn.CTCLoss(blank=0, reduction='mean')
targets = torch.randint(1, 50, (4, 20))  # [B, S]
input_lengths = torch.full((4,), 100)
target_lengths = torch.full((4,), 20)

loss = ctc_loss(log_probs.transpose(0, 1), targets, input_lengths, target_lengths)
print(f'CTC Loss: {loss.item():.4f}')

In [ ]:
def ctc_greedy_decode(log_probs, blank_id=0):
    """CTC 贪婪解码"""
    predictions = log_probs.argmax(dim=-1)  # [B, T]
    decoded = []
    
    for pred in predictions:
        # 去除重复和空白
        result = []
        prev = blank_id
        for p in pred:
            if p != blank_id and p != prev:
                result.append(p.item())
            prev = p
        decoded.append(result)
    
    return decoded

# 测试解码
decoded = ctc_greedy_decode(log_probs)
print(f'解码结果长度: {[len(d) for d in decoded]}')

## 2. Conformer 架构

Conformer = CNN + Transformer，结合局部和全局建模能力。

In [ ]:
class ConvModule(nn.Module):
    """Conformer 卷积模块"""
    def __init__(self, dim, kernel_size=31):
        super().__init__()
        self.layers = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 2),
            nn.GLU(dim=-1),
        )
        self.depthwise = nn.Conv1d(dim, dim, kernel_size, 
                                   padding=kernel_size//2, groups=dim)
        self.norm = nn.BatchNorm1d(dim)
        self.pointwise = nn.Linear(dim, dim)
    
    def forward(self, x):
        x = self.layers(x)
        x = x.transpose(1, 2)
        x = self.depthwise(x)
        x = self.norm(x)
        x = F.silu(x)
        x = x.transpose(1, 2)
        return self.pointwise(x)

class ConformerBlock(nn.Module):
    """Conformer 块: FFN → MHSA → Conv → FFN"""
    def __init__(self, dim=256, num_heads=4, ff_mult=4):
        super().__init__()
        self.ffn1 = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim*ff_mult),
                                  nn.SiLU(), nn.Linear(dim*ff_mult, dim))
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.attn_norm = nn.LayerNorm(dim)
        self.conv = ConvModule(dim)
        self.ffn2 = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim*ff_mult),
                                  nn.SiLU(), nn.Linear(dim*ff_mult, dim))
        self.norm = nn.LayerNorm(dim)
    
    def forward(self, x):
        x = x + 0.5 * self.ffn1(x)
        x_norm = self.attn_norm(x)
        x = x + self.attn(x_norm, x_norm, x_norm)[0]
        x = x + self.conv(x)
        x = x + 0.5 * self.ffn2(x)
        return self.norm(x)

# 测试
conformer = ConformerBlock().to(device)
x = torch.randn(4, 100, 256).to(device)
out = conformer(x)
print(f'Conformer 输出: {out.shape}')

## 3. 非自回归 TTS (FastSpeech 风格)

In [ ]:
class VariancePredictor(nn.Module):
    """预测时长/音高/能量"""
    def __init__(self, dim=256):
        super().__init__()
        self.conv1 = nn.Conv1d(dim, dim, 3, padding=1)
        self.conv2 = nn.Conv1d(dim, dim, 3, padding=1)
        self.linear = nn.Linear(dim, 1)
    
    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.transpose(1, 2)
        return self.linear(x).squeeze(-1)

class LengthRegulator(nn.Module):
    """根据时长扩展序列"""
    def forward(self, x, durations):
        # durations: [B, T] 每个位置的持续帧数
        output = []
        for i in range(x.size(0)):
            expanded = torch.repeat_interleave(x[i], durations[i].long(), dim=0)
            output.append(expanded)
        return torch.nn.utils.rnn.pad_sequence(output, batch_first=True)

# 测试
duration_pred = VariancePredictor().to(device)
x = torch.randn(2, 20, 256).to(device)  # 文本编码
durations = duration_pred(x).exp().clamp(1, 20)  # 预测时长
print(f'预测时长: {durations.shape}')
print(f'时长范围: {durations.min():.1f} - {durations.max():.1f}')

## 4. 音频数据增强

In [ ]:
class AudioAugmentation:
    """音频数据增强"""
    @staticmethod
    def add_noise(audio, snr_db=10):
        """添加高斯噪声"""
        signal_power = (audio ** 2).mean()
        noise_power = signal_power / (10 ** (snr_db / 10))
        noise = torch.randn_like(audio) * noise_power.sqrt()
        return audio + noise
    
    @staticmethod
    def time_mask(spec, max_mask=50):
        """时间遮蔽"""
        t = torch.randint(1, max_mask, (1,)).item()
        t0 = torch.randint(0, max(1, spec.size(-1) - t), (1,)).item()
        spec[..., t0:t0+t] = 0
        return spec
    
    @staticmethod
    def freq_mask(spec, max_mask=20):
        """频率遮蔽"""
        f = torch.randint(1, max_mask, (1,)).item()
        f0 = torch.randint(0, max(1, spec.size(-2) - f), (1,)).item()
        spec[..., f0:f0+f, :] = 0
        return spec

# 测试
spec = torch.randn(1, 80, 100)
aug = AudioAugmentation()

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].imshow(spec[0], aspect='auto', origin='lower')
axes[0].set_title('Original')

spec_time = aug.time_mask(spec.clone())
axes[1].imshow(spec_time[0], aspect='auto', origin='lower')
axes[1].set_title('Time Mask')

spec_freq = aug.freq_mask(spec.clone())
axes[2].imshow(spec_freq[0], aspect='auto', origin='lower')
axes[2].set_title('Freq Mask')

plt.tight_layout()
plt.show()

## 5. 评估指标

In [ ]:
def word_error_rate(reference, hypothesis):
    """词错误率 (WER)"""
    ref = reference.split()
    hyp = hypothesis.split()
    
    # 动态规划
    d = [[0] * (len(hyp) + 1) for _ in range(len(ref) + 1)]
    for i in range(len(ref) + 1): d[i][0] = i
    for j in range(len(hyp) + 1): d[0][j] = j
    
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            if ref[i-1] == hyp[j-1]:
                d[i][j] = d[i-1][j-1]
            else:
                d[i][j] = min(d[i-1][j], d[i][j-1], d[i-1][j-1]) + 1
    
    return d[len(ref)][len(hyp)] / len(ref) if ref else 0

# 测试
ref = "hello world how are you"
hyp = "hello word how are you"  # 'world' -> 'word'
wer = word_error_rate(ref, hyp)
print(f'Reference: {ref}')
print(f'Hypothesis: {hyp}')
print(f'WER: {wer:.2%}')

## 6. 流式推理

In [ ]:
class StreamingASR:
    """流式语音识别"""
    def __init__(self, chunk_size=16000, overlap=4000):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.buffer = []
    
    def process_chunk(self, audio_chunk):
        """处理音频块"""
        self.buffer.append(audio_chunk)
        total = torch.cat(self.buffer, dim=-1)
        
        if total.size(-1) >= self.chunk_size:
            # 处理并保留重叠
            to_process = total[..., :self.chunk_size]
            self.buffer = [total[..., self.chunk_size - self.overlap:]]
            return to_process
        return None

# 测试
streamer = StreamingASR(chunk_size=8000, overlap=2000)
for i in range(5):
    chunk = torch.randn(1, 2000)
    result = streamer.process_chunk(chunk)
    if result is not None:
        print(f'Chunk {i}: 处理 {result.shape[-1]} 样本')
    else:
        print(f'Chunk {i}: 缓冲中...')

## 总结

| 主题 | 关键技术 | 应用 |
|:-----|:---------|:-----|
| CTC | 序列对齐 | 端到端 ASR |
| Conformer | CNN+Transformer | 高精度 ASR |
| FastSpeech | 非自回归 | 快速 TTS |
| SpecAugment | 频谱遮蔽 | 数据增强 |
| WER/CER | 编辑距离 | ASR 评估 |